# Scope Data Blueprint (Polars)

Ce notebook synthétise les besoins "Scope & Données" partagés (Retailers, Publishers, segments Hill's, Clean Room) à partir des fichiers `input_data/X_2025`. L'objectif est double :

- **Cartographier** les jeux de données disponibles pour chacun des blocs demandés (SKU/prix, expositions média, segments CRM, jointures anonymisées, etc.).
- **Produire des KPI prêts à l'emploi** et des structures de données réutilisables (Polars) pour alimenter des analyses métier ou des exports vers un clean room.

> **Technologie** : toutes les manipulations sont effectuées en Polars (lazy evaluation) pour rester aligné avec la stratégie de performance du projet.


## 0. Périmètre attendu

| Bloc | Besoins métier | Sources utilisées |
| --- | --- | --- |
| **Retailers** | Transactions, SKU, prix, historique client, fréquence/panier, ruptures & stock | `retailer.csv` |
| **Publishers** | Impressions/clicks, ordre d'exposition par ID, interactions (watchtime/CTR), données socio-démo | `tv_publisher.csv`, `programmatic_publisher.csv`, `mapping_transac_publisher_tv.csv`, `socio_demo.csv` |
| **Hill's / CRM** | Segments CRM, mix produits (Prescription Diet, Science Diet, …), feedback clients | `retailer.csv`, `socio_demo.csv` |
| **Clean Room** | Jointure anonymisée Retail × Publisher × Hill's, séquences d'exposition, lift incrémental (closed-loop) | Ensemble des fichiers via Polars + mapping |

Chaque section ci-dessous construit les structures de données nécessaires et documente les éventuels manques (ex. absence de colonnes `stock_level`, `clicks`, `watch_time`).


In [13]:
# Imports & configuration
import polars as pl
import pandas as pd
from pathlib import Path

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(12)

DATA_PATH = Path("input_data") / "X_2025"
print(f"Données chargées depuis: {DATA_PATH.resolve()}")


Données chargées depuis: /home/nathan/Documents/data_challenge_mie/input_data/X_2025


In [14]:
# Chargement lazy des datasets (on force les dates à rester en String)
sc_retailer = pl.scan_csv(DATA_PATH / "retailer.csv")
sc_tv = pl.scan_csv(DATA_PATH / "tv_publisher.csv")
sc_prog = pl.scan_csv(DATA_PATH / "programmatic_publisher.csv")
sc_mapping = pl.scan_csv(DATA_PATH / "mapping_transac_publisher_tv.csv")
sc_socio = pl.scan_csv(DATA_PATH / "socio_demo.csv")

datasets = {
    "Retailer": sc_retailer,
    "TV": sc_tv,
    "Programmatic": sc_prog,
    "Mapping": sc_mapping,
    "SocioDemo": sc_socio,
}

for name, lf in datasets.items():
    print("\n" + "-" * 80)
    print(f"{name} → colonnes: {list(lf.schema.keys())}")
    sample = lf.head(3).collect()
    print(sample)
print("\nLazyFrames prêts pour les jointures Polars.")



--------------------------------------------------------------------------------
Retailer → colonnes: ['customer_id', 'timestamp_utc', 'event_name', 'brand', 'product_name', 'sales', 'quantity']
shape: (3, 7)
┌────────────────┬───────────────┬───────────────┬──────────────┬───────────────┬───────┬──────────┐
│ customer_id    ┆ timestamp_utc ┆ event_name    ┆ brand        ┆ product_name  ┆ sales ┆ quantity │
│ ---            ┆ ---           ┆ ---           ┆ ---          ┆ ---           ┆ ---   ┆ ---      │
│ str            ┆ str           ┆ str           ┆ str          ┆ str           ┆ f64   ┆ f64      │
╞════════════════╪═══════════════╪═══════════════╪══════════════╪═══════════════╪═══════╪══════════╡
│ reFs5GI87lXJkJ ┆ 2024-02-07    ┆ Product Page  ┆ null         ┆ null          ┆ null  ┆ null     │
│ Si9r           ┆ 02:27:10      ┆ View          ┆              ┆               ┆       ┆          │
│ reFs5GI87lXJkJ ┆ 2024-06-12    ┆ Product Page  ┆ Science Diet ┆ SD Fel A7+    ┆ n

/tmp/ipykernel_34554/1122510760.py:18: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  print(f"{name} → colonnes: {list(lf.schema.keys())}")


## 1. Données Retailers (Transactions / SKU / Prix / Historique)

Objectifs :
- **Transactions & paniers** : volume, panier moyen, fréquence d'achat par client.
- **SKU & prix** : couverture assortiment, prix moyen par produit.
- **Historique & fréquence** : timeline hebdomadaire/mois + récurrence clients.
- **Ruptures & stock** : identification des transactions avec `quantity ≤ 0` comme proxy d'anomalie (le dataset ne contient pas de colonnes explicites `stock_level`).


In [15]:
# Préparation Retailer (parse → cache eager → réutilisation lazy)
retailer_base = (
    sc_retailer
    .with_columns([
        pl.col("timestamp_utc").str.strptime(pl.Datetime, strict=False, format=None).alias("ts"),
        pl.col("sales").fill_null(0),
        pl.col("quantity").fill_null(0),
    ])
    .with_columns([
        pl.col("ts").dt.date().alias("order_date"),
        pl.col("ts").dt.truncate("1w").alias("week_start"),
        pl.col("ts").dt.truncate("1mo").alias("month_start"),
        pl.when(pl.col("quantity") <= 0)
          .then(None)
          .otherwise(pl.col("sales") / pl.col("quantity"))
          .alias("unit_price"),
    ])
)

retailer = retailer_base.collect(streaming=True)
retailer_lazy = retailer.lazy()

retailer_kpis = (
    retailer_lazy
    .select([
        pl.len().alias("transactions"),
        pl.col("customer_id").n_unique().alias("unique_customers"),
        pl.col("product_name").n_unique().alias("unique_skus"),
        pl.col("sales").sum().alias("total_sales"),
        pl.col("sales").mean().alias("avg_basket"),
        (pl.col("sales").sum() / pl.col("quantity").sum()).alias("avg_unit_price"),
        pl.col("quantity").mean().alias("avg_units_per_tx"),
    ])
    .collect()
)

customer_lifecycle = (
    retailer_lazy
    .group_by("customer_id")
    .agg([
        pl.len().alias("transactions"),
        pl.col("sales").sum().alias("sales"),
        pl.col("ts").min().alias("first_purchase"),
        pl.col("ts").max().alias("last_purchase"),
    ])
    .with_columns([
        (pl.col("last_purchase") - pl.col("first_purchase")).dt.total_days().alias("lifetime_days"),
    ])
)

customer_stats = (
    customer_lifecycle
    .select([
        pl.len().alias("customers"),
        pl.col("transactions").mean().alias("avg_tx_per_customer"),
        pl.col("sales").mean().alias("avg_customer_value"),
        pl.col("lifetime_days").mean().alias("avg_lifetime_days"),
        pl.col("lifetime_days").median().alias("median_lifetime_days"),
    ])
    .collect()
)

weekly_history = (
    retailer_lazy
    .group_by("week_start")
    .agg([
        pl.col("sales").sum().alias("revenue"),
        pl.len().alias("transactions"),
        pl.col("customer_id").n_unique().alias("active_customers"),
    ])
    .sort("week_start")
    .collect()
)

sku_pricing = (
    retailer_lazy
    .group_by("product_name")
    .agg([
        pl.col("sales").sum().alias("revenue"),
        pl.col("quantity").sum().alias("units"),
        pl.col("unit_price").mean().alias("avg_unit_price"),
    ])
    .sort("revenue", descending=True)
    .head(25)
    .collect()
)

stock_alerts = (
    retailer_lazy
    .filter(pl.col("quantity") <= 0)
    .group_by("product_name")
    .agg([
        pl.len().alias("anomaly_tx"),
        pl.col("sales").sum().alias("sales_flagged"),
    ])
    .sort("anomaly_tx", descending=True)
    .collect()
)

print("KPIs Retailer :")
print(retailer_kpis)
print("\nStats clients :")
print(customer_stats)
print("\nHistorique hebdomadaire (aperçu) :")
print(weekly_history.head(12))
print("\nTop 25 SKU / prix moyen :")
print(sku_pricing)
print("\nTransactions suspectes (quantity ≤ 0) → proxy ruptures/retours :")
print(stock_alerts)


/tmp/ipykernel_34554/3321825689.py:20: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  retailer = retailer_base.collect(streaming=True)


KPIs Retailer :
shape: (1, 7)
┌──────────────┬──────────────┬─────────────┬─────────────┬────────────┬─────────────┬─────────────┐
│ transactions ┆ unique_custo ┆ unique_skus ┆ total_sales ┆ avg_basket ┆ avg_unit_pr ┆ avg_units_p │
│ ---          ┆ mers         ┆ ---         ┆ ---         ┆ ---        ┆ ice         ┆ er_tx       │
│ u32          ┆ ---          ┆ u32         ┆ f64         ┆ f64        ┆ ---         ┆ ---         │
│              ┆ u32          ┆             ┆             ┆            ┆ f64         ┆ f64         │
╞══════════════╪══════════════╪═════════════╪═════════════╪════════════╪═════════════╪═════════════╡
│ 9866049      ┆ 1354584      ┆ 506         ┆ 6.2428e7    ┆ 6.327553   ┆ 40.120764   ┆ 0.157713    │
└──────────────┴──────────────┴─────────────┴─────────────┴────────────┴─────────────┴─────────────┘

Stats clients :
shape: (1, 5)
┌───────────┬─────────────────────┬────────────────────┬───────────────────┬──────────────────────┐
│ customers ┆ avg_tx_per_custom

## 2. Données Publishers (Impressions, Ordre d'exposition, Interactions)

Composants couverts :
- **TV & Programmatique** : conversion des `cost_milli_cent` en dollars, comptage des impressions, CPM et reach.
- **Ordre d'exposition** : séquences chronologiques par `customer_id` (via mapping device/dsp → client).
- **Interactions** : pas de colonnes `clicks` / `watch_time` dans les CSV fournis → la cellule produit un drapeau `data_gap` pour ces métriques.
- **Enrichissement socio-démo** : jonction avec `socio_demo` pour savoir quels segments reçoivent quelles expositions.


In [ ]:
# Préparation Publishers
mapping_devices = sc_mapping.select(["customer_id", "device_id"]).filter(pl.col("device_id").is_not_null())
mapping_dsp = sc_mapping.select(["customer_id", "dsp_id"]).filter(pl.col("dsp_id").is_not_null())

common_projection = ["customer_id", "touchpoint_id", "timestamp", "spend_usd", "channel"]

tv_events = (
    sc_tv
    .with_columns([
        pl.col("timestamp_utc").str.strptime(pl.Datetime, strict=False).alias("timestamp"),
        (pl.col("cost_milli_cent") / 100000).alias("spend_usd"),
    ])
    .join(mapping_devices, on="device_id", how="inner")
    .select([
        pl.col("customer_id"),
        pl.col("device_id").alias("touchpoint_id"),
        pl.col("timestamp"),
        pl.col("spend_usd"),
        pl.lit("TV").alias("channel"),
    ])
)

prog_events = (
    sc_prog
    .with_columns([
        pl.col("timestamp_utc").str.strptime(pl.Datetime, strict=False).alias("timestamp"),
        (pl.col("cost_milli_cent") / 100000).alias("spend_usd"),
    ])
    .join(mapping_dsp, on="dsp_id", how="inner")
    .select([
        pl.col("customer_id"),
        pl.col("dsp_id").alias("touchpoint_id"),
        pl.col("timestamp"),
        pl.col("spend_usd"),
        pl.lit("Programmatic").alias("channel"),
    ])
)

publisher_events = pl.concat([tv_events, prog_events], how="vertical")

publisher_kpis = (
    publisher_events
    .group_by("channel")
    .agg([
        pl.len().alias("impressions"),
        pl.col("customer_id").n_unique().alias("unique_customers"),
        pl.col("spend_usd").sum().alias("spend_usd"),
        (pl.col("spend_usd").sum() / pl.len() * 1000).alias("cpm"),
    ])
    .collect()
)

sequence_sample = (
    publisher_events
    .sort(["customer_id", "timestamp"])
    .group_by("customer_id")
    .agg([
        pl.len().alias("touchpoints"),
        pl.col("channel").alias("channel_sequence"),
        pl.col("timestamp").alias("timestamps"),
    ])
    .filter(pl.col("touchpoints") >= 3)
    .limit(5)
    .collect()
)

publisher_socio = (
    publisher_events
    .join(sc_socio, on="customer_id", how="left")
    .group_by(["channel", "age", "income"])
    .agg([
        pl.col("customer_id").n_unique().alias("unique_customers"),
        pl.len().alias("impressions"),
    ])
    .sort("impressions", descending=True)
    .head(20)
    .collect()
)

publisher_gaps = pl.DataFrame({
    "metric": ["clicks", "watch_time"],
    "status": ["missing", "missing"],
    "comment": [
        "Aucune colonne de clics fournie → prévoir pixel ou log ad-server.",
        "Pas de données watch_time → nécessaire pour la vidéo.",
    ],
})

print("KPIs publishers :")
print(publisher_kpis)
print("\nSéquences d'exposition (échantillon) :")
print(sequence_sample)
print("\nRépartition socio-démo des expositions (top 20 combinaisons) :")
print(publisher_socio)
print("\nDonnées manquantes côté interactions :")
print(publisher_gaps)


AttributeError: 'LazyFrame' object has no attribute 'vstack'